# Research-grounded BoC forecast

Inspect the documents visible at one historical forecast origin and the exact agent prompt. The live model call is opt-in so opening or running the notebook does not accidentally incur cost.


Run the following steps when once when Coder is launched
1. `uv sync`
2. `uv run pytest implementations/tests/boc_rate_decisions/test_research.py -v`
3. `uv run pytest implementations/tests/boc_rate_decisions -v`
4. `uv run python scripts/fetch_boc.py`
5. `uv run python scripts/fetch_boc_press_releases.py`

To run this code via CLI, execute the following code:
1. `uv run jupyter lab implementations/boc_rate_decisions/04_research_grounded_forecast.ipynb`
2. `uv run python implementations/boc_rate_decisions/run_research_agent.py --live` -- one live prediction with default model
2. `uv run python implementations/boc_rate_decisions/run_research_agent.py --compare` -- deterministic comparision between methods
3. `uv run python implementations/boc_rate_decisions/run_research_agent.py --compare --include-agents --model gemini-3.1-flash-lite-preview` -- full comparision with model calls

In [1]:
import json
from datetime import datetime
from pathlib import Path

import yaml
from IPython.display import JSON, Markdown, display
from aieng.forecasting.evaluation import BacktestSpec
from boc_rate_decisions.analyst_agent import (
    BoCDecisionPromptBuilder,
    build_boc_research_predictor,
)
from boc_rate_decisions.data import build_boc_service
from boc_rate_decisions.research import DEFAULT_RESEARCH_SOURCES, format_research_evidence


## Configuration

The default origin is 28 days before the June 5, 2024 decision. `AGENT_MODEL` controls every agent invocation in the notebook. The lite model is the project default; uncomment the advanced alternative for a more capable, more expensive run. Set `RUN_LIVE = True` only when model credentials are configured and you intend to make a billed model call.


In [2]:
def find_repo_root(start: Path | None = None) -> Path:
    candidate = (start or Path.cwd()).resolve()
    for directory in (candidate, *candidate.parents):
        if (directory / "implementations/boc_rate_decisions").is_dir() and (directory / "scripts").is_dir():
            return directory
    raise FileNotFoundError(f"Could not locate the repository root from {candidate}")

REPO_ROOT = find_repo_root()
ORIGIN = datetime(2024, 5, 8)
REPORTS_DIR = REPO_ROOT / "data/reports/boc_press_releases"
SPEC_PATH = REPO_ROOT / "implementations/boc_rate_decisions/specs/boc_rate_direction_smoke.yaml"
MAX_DOCUMENTS = 3
MAX_CHARS_PER_DOCUMENT = 6_000
# AGENT_MODEL = "gemini-3.1-flash-lite-preview"
# AGENT_MODEL = "gemini-3.5-flash"  # Gemini advance model
AGENT_MODEL = "claude-opus-5"     # Claude advance model
RUN_LIVE = True

print(f"Repository root: {REPO_ROOT}")
print(f"Reports directory: {REPORTS_DIR}")
print(f"Agent model: {AGENT_MODEL}")
if not REPORTS_DIR.is_dir():
    raise FileNotFoundError(
        f"BoC release cache not found at {REPORTS_DIR}. "
        "Run from the repository root: uv run python scripts/fetch_boc_press_releases.py --year 2024"
    )


Repository root: /home/coder/agentic-forecasting
Reports directory: /home/coder/agentic-forecasting/data/reports/boc_press_releases
Agent model: claude-opus-5


## Load the cutoff-scoped data context


In [3]:
with SPEC_PATH.open(encoding="utf-8") as file:
    smoke_spec = BacktestSpec.model_validate(yaml.safe_load(file))
task = smoke_spec.task

service = build_boc_service(reports_dir=REPORTS_DIR)
context = service.context(ORIGIN)
print(f"Origin: {ORIGIN.date()} | Task: {task.task_id}")


Origin: 2024-05-08 | Task: boc_rate_direction_next_meeting


## Cutoff-visible evidence

Every displayed document must have a publication date on or before the origin.


In [4]:
evidence = format_research_evidence(
    context,
    sources=DEFAULT_RESEARCH_SOURCES,
    max_documents=MAX_DOCUMENTS,
    max_chars_per_document=MAX_CHARS_PER_DOCUMENT,
)
assert all(item["publication_date"] <= ORIGIN.date().isoformat() for item in evidence)
display(JSON(evidence, expanded=False))


<IPython.core.display.JSON object>

## Generated prompt

This is the exact JSON prompt that will be sent to the research-grounded analyst. Inspect it before enabling the model call.


In [5]:
builder = BoCDecisionPromptBuilder(
    document_sources=DEFAULT_RESEARCH_SOURCES,
    max_documents=MAX_DOCUMENTS,
    max_chars_per_document=MAX_CHARS_PER_DOCUMENT,
)
prompt = builder(task=task, context=context)
display(JSON(json.loads(prompt), expanded=False))


<IPython.core.display.JSON object>

## Optional live model prediction

Change `RUN_LIVE` to `True` in the configuration cell to invoke the configured model. The result is rendered as Markdown with probabilities, rationale, and key signals.


In [6]:
if RUN_LIVE:
    predictor = build_boc_research_predictor(
        model=AGENT_MODEL,
        max_documents=MAX_DOCUMENTS,
        max_chars_per_document=MAX_CHARS_PER_DOCUMENT,
    )
    predictions = predictor.predict(task, context)
    if not predictions:
        raise RuntimeError("The research agent returned no prediction.")

    prediction = predictions[0]
    probabilities = prediction.payload.probabilities
    rationale = prediction.metadata.get("rationale", "No rationale returned.")
    signals = prediction.metadata.get("key_signals", [])
    rows = "\n".join(f"| {label} | {probability:.1%} |" for label, probability in probabilities.items())
    signal_text = "\n".join(f"- {signal}" for signal in signals) or "- None returned"
    display(Markdown(
        f"### Forecast for {prediction.forecast_date:%Y-%m-%d}\n\n"
        f"| Decision | Probability |\n|---|---:|\n{rows}\n\n"
        f"**Rationale:** {rationale}\n\n**Key signals:**\n{signal_text}"
    ))
else:
    display(Markdown("> Live prediction skipped. Set `RUN_LIVE = True` in the configuration cell to call the model."))

/home/coder/agentic-forecasting/.venv/lib/python3.12/site-packages/google/adk/tools/function_tool.py:95: UserWarning: [EXPERIMENTAL] feature FeatureName.JSON_SCHEMA_FOR_FUNC_DECL is enabled.
  build_function_declaration(


### Forecast for 2024-06-05

| Decision | Probability |
|---|---:|
| cut | 53.0% |
| hold | 46.0% |
| hike | 1.0% |

**Rationale:** Setup: the policy rate has been parked at 5.00% since July 2023 with six consecutive holds (2023-09 through 2024-04-10), so the Bank is at the end of a tightening cycle and squarely in a 'when do we start easing' debate rather than a tightening one. A hike is effectively off the table: the last move was a hike ten months ago, the economy is described by the Bank itself as being in modest excess supply, and no reversal to tightening from a long hold string with falling inflation is plausible at a single meeting.

Documentary evidence (research_evidence, three Bank press releases): the language has shifted materially and monotonically dovish across January -> March -> April. 2024-01-24_en said core measures 'are not showing sustained declines' and Council was 'still concerned about risks... particularly the persistence in underlying inflation.' 2024-03-06_en repeated that concern but noted CPI eased to 2.9% and wage pressures 'may be easing.' 2024-04-10_en drops the 'still concerned about... persistence' sentence entirely, notes CPI at 2.8% in February with easing 'becoming more broad-based,' core measures 'slowed to just over 3%' with '3-month annualized rates... suggesting downward momentum,' unemployment rising to 6.1%, and states Council 'will be looking for evidence that this downward momentum is sustained.' That is the Bank's standard pre-pivot formulation: an explicit, near-term, verifiable condition for cutting rather than an open-ended concern. It also flagged the next full MPR only on July 24, confirming that June 5 is a non-MPR meeting — the Bank has historically been willing to start moves at non-MPR dates (e.g., 2015-01-21), so the absence of an MPR is only a mild deterrent.

Macro snapshot corroborates: inflation gap of ~+0.90pp means headline CPI is near 2.9%, well down from the 2023 peaks and inside the Bank's control range; unemployment momentum of +1.1pp confirms a clearly loosening labour market (the Bank's April statement cites 6.1%); and the 2-year GoC trading 85bp below the 5% policy rate is an unambiguous market pricing of a near-term easing cycle of roughly three-plus cuts over the next year — historically that magnitude of inversion has preceded a first cut within a couple of meetings, and the Bank is reluctant to disappoint a bond market that far ahead only if the data cooperate.

Why not higher than ~0.53 for a cut: the decisive data at this 28-day lead are still unobserved. The April CPI print (~May 21) and Q1 GDP (~May 31) both land after this forecast origin and both are explicitly the 'evidence that this downward momentum is sustained' the Bank said it wanted. One firm core CPI print, or a GDP/wage upside surprise, would tip Governing Council to wait for the July MPR, and the Bank's institutional gradualism plus its repeated insistence on wanting *further and sustained* easing (Jan and Mar statements) argues for a genuine coin-flip rather than a confident call. Governor Macklem had signalled only that a June cut was within the realm of possibility, not that it was the base case. Historical base rates (hold 80%) also pull toward hold, but they are dominated here by the regime signal: cuts cluster at the start of easing cycles, and every conditioning variable says we are at that inflection.

Net: a slight tilt to cut over hold, with hike as a rounding error.

**Key signals:**
- 2024-04-10_en (Bank statement): dropped the 'still concerned about the persistence in underlying inflation' language used in 2024-01-24_en and 2024-03-06_en, and instead said Council 'will be looking for evidence that this downward momentum is sustained' — a conditional, near-term pivot signal
- 2024-04-10_en (Bank statement): core inflation 'slowed to just over 3% in February, and 3-month annualized rates are suggesting downward momentum'; CPI 2.8% with easing 'more broad-based'; forecast to move below 2.5% in H2 2024
- 2024-04-10_en (Bank statement): unemployment rate risen gradually to 6.1% in March, employment growing slower than working-age population, 'some recent signs that wage pressures are moderating' — corroborated by macro snapshot unemployment momentum +1.1pp
- Macro snapshot: 2-year GoC yield 85bp BELOW the 5% policy rate — bond market pricing an imminent and sustained easing cycle
- Macro snapshot: inflation gap ~+0.90pp (CPI ~2.9%), inside the Bank's 1-3% control range for the first sustained stretch since 2021
- Six consecutive holds at 5.00% since July 2023 with zero rate momentum — end-of-tightening regime; direct hold-to-hike reversal essentially impossible given excess supply
- My inference (not a Bank statement): the April CPI (~May 21) and Q1 GDP (~May 31) prints land after the forecast origin and are the literal 'evidence' the Bank conditioned on, which caps confidence in a cut at roughly a coin flip at this 28-day lead
- 2024-04-10_en information note: next full MPR is July 24, so June 5 is a non-MPR meeting — a mild but not decisive argument for waiting

---
# Scientific comparison across historical decisions

The first half verifies one forecast's evidence pipeline. This section tests whether research grounding improves probabilistic forecasts across the smoke spec's three historical decisions. It compares historical frequencies, logistic regression, a quantitative-only agent, a research-grounded agent, and a market-expectations proxy.

> **Market-data limitation:** the repository has the two-year Government of Canada yield, not historical meeting-specific CORRA/OIS probabilities. The market method below is a transparent yield-spread proxy—not a literal market-implied probability. Replace it with archived OIS data before claiming market-relative skill.


In [7]:
from datetime import timezone

import numpy as np
import pandas as pd
from aieng.forecasting.evaluation import CategoricalForecast, Prediction, Predictor, backtest
from aieng.forecasting.methods import CategoricalFrequencyPredictor
from boc_rate_decisions.analysis import score_leaderboard
from boc_rate_decisions.analyst_agent import build_boc_agent_predictor, build_boc_basic_config
from boc_rate_decisions.data import BOND_YIELD_2YR_SERIES_ID, TARGET_RATE_SERIES_ID
from boc_rate_decisions.predictors.logistic_baseline import BoCLogisticPredictor


## Define the market-expectations proxy

A two-year yield below the policy rate shifts probability toward cuts; a yield above it shifts probability toward hikes; a small spread favours a hold. The fixed mapping is not tuned on the three evaluation outcomes.


In [8]:
class TwoYearYieldMarketProxy(Predictor):
    @property
    def predictor_id(self) -> str:
        return "boc_two_year_yield_market_proxy"

    def predict(self, task, context):
        rate = float(context.get_series(TARGET_RATE_SERIES_ID)["value"].iloc[-1])
        yield_2y = float(context.get_series(BOND_YIELD_2YR_SERIES_ID)["value"].iloc[-1])
        spread = yield_2y - rate
        scaled = spread / 0.25
        scores = np.array([-scaled, 1.5 - abs(scaled), scaled], dtype=float)
        weights = np.exp(scores - scores.max())
        probabilities = weights / weights.sum()
        labels = [category.label for category in task.categories]
        payload = CategoricalForecast(probabilities=dict(zip(labels, probabilities, strict=True)))
        offset = pd.tseries.frequencies.to_offset(task.frequency)
        forecast_date = pd.Timestamp(context.as_of) + offset * task.horizons[0]
        return [Prediction(
            predictor_id=self.predictor_id,
            task_id=task.task_id,
            issued_at=datetime.now(tz=timezone.utc).replace(tzinfo=None),
            as_of=context.as_of,
            forecast_date=forecast_date.to_pydatetime(),
            payload=payload,
            metadata={"yield_2y": yield_2y, "policy_rate": rate, "yield_spread": spread},
        )]


## Run all three historical origins

The deterministic methods always run. Set `RUN_AGENT_BACKTESTS = True` to add six calls to the model selected by `AGENT_MODEL`: three for each agent. This flag is separate from the earlier single-prediction flag.


In [9]:
RUN_AGENT_BACKTESTS = True

comparison_predictors = {
    "historical_frequency": CategoricalFrequencyPredictor(),
    "logistic_regression": BoCLogisticPredictor(),
    "two_year_market_proxy": TwoYearYieldMarketProxy(),
}
if RUN_AGENT_BACKTESTS:
    comparison_predictors["quantitative_only_agent"] = build_boc_agent_predictor(
        build_boc_basic_config(model=AGENT_MODEL)
    )
    comparison_predictors["research_grounded_agent"] = build_boc_research_predictor(
        model=AGENT_MODEL,
        max_documents=MAX_DOCUMENTS,
        max_chars_per_document=MAX_CHARS_PER_DOCUMENT,
    )

comparison_results = {}
for name, method in comparison_predictors.items():
    print(f"Running {name}...")
    comparison_results[name] = backtest(method, smoke_spec, service)
print("Comparison complete.")


Running historical_frequency...
Running logistic_regression...
Running two_year_market_proxy...
Running quantitative_only_agent...
Running research_grounded_agent...
Comparison complete.


## RPS leaderboard

Lower Ranked Probability Score is better. Positive skill means improvement over historical frequencies. Three decisions are enough to verify the pipeline, but not enough for a reliable ranking.


In [10]:
board = score_leaderboard(comparison_results, reference_id="historical_frequency")
display(board)

,predictor_id,metric,mean_score,n_predictions,n_skipped_origins,skill_vs_reference
0,quantitative_only_agent,rps,0.071905,3,0,0.8832
1,research_grounded_agent,rps,0.082908,3,0,0.8654
2,two_year_market_proxy,rps,0.330410,3,0,0.4634
3,logistic_regression,rps,0.368007,3,0,0.4023
4,historical_frequency,rps,0.615751,3,0,0.0000


## Meeting-level comparison

This table places the model's most likely decision beside the actual decision. `correct` evaluates the point prediction, while RPS evaluates the full probability distribution. A method can choose the correct outcome but still receive a poor RPS if it was badly calibrated or put too much probability on the opposite tail.


In [11]:
resolved = service.get_series(task.target_series_id, as_of=datetime.now())
value_to_label = {category.value: category.label for category in task.categories}
outcome_by_date = {pd.Timestamp(ts).date(): value_to_label[float(value)] for ts, value in zip(resolved["timestamp"], resolved["value"], strict=True)}

rows = []
for method_name, result in comparison_results.items():
    for prediction, rps in zip(result.predictions, result.scores, strict=True):
        meeting = pd.Timestamp(prediction.forecast_date).date()
        probabilities = prediction.payload.probabilities
        predicted_outcome = max(probabilities, key=probabilities.get)
        actual_outcome = outcome_by_date.get(meeting)
        rows.append({
            "method": method_name,
            "origin": pd.Timestamp(prediction.as_of).date(),
            "meeting": meeting,
            "predicted_outcome": predicted_outcome,
            "actual_outcome": actual_outcome,
            "correct": predicted_outcome == actual_outcome,
            "prediction_confidence": probabilities[predicted_outcome],
            **{f"p_{label}": probability for label, probability in probabilities.items()},
            "rps": rps,
        })

meeting_comparison = pd.DataFrame(rows).sort_values(["meeting", "method"])
display(meeting_comparison.style.format({
    "prediction_confidence": "{:.1%}",
    "p_cut": "{:.1%}", "p_hold": "{:.1%}",
    "p_hike": "{:.1%}", "rps": "{:.3f}",
}))


,method,origin,meeting,predicted_outcome,actual_outcome,correct,prediction_confidence,p_cut,p_hold,p_hike,rps
0,historical_frequency,2024-03-13,2024-04-10,hold,hold,True,80.3%,4.9%,80.3%,14.8%,0.024
3,logistic_regression,2024-03-13,2024-04-10,hold,hold,True,94.8%,4.2%,94.8%,1.0%,0.002
9,quantitative_only_agent,2024-03-13,2024-04-10,hold,hold,True,92.0%,7.0%,92.0%,1.0%,0.005
12,research_grounded_agent,2024-03-13,2024-04-10,hold,hold,True,90.0%,9.0%,90.0%,1.0%,0.008
6,two_year_market_proxy,2024-03-13,2024-04-10,cut,hold,False,99.6%,99.6%,0.4%,0.1%,0.991
1,historical_frequency,2024-05-08,2024-06-05,hold,cut,False,80.5%,4.9%,80.5%,14.6%,0.926
4,logistic_regression,2024-05-08,2024-06-05,hold,cut,False,95.4%,3.5%,95.4%,1.1%,0.931
10,quantitative_only_agent,2024-05-08,2024-06-05,cut,cut,True,55.0%,55.0%,44.0%,1.0%,0.203
13,research_grounded_agent,2024-05-08,2024-06-05,cut,cut,True,52.0%,52.0%,47.0%,1.0%,0.230
7,two_year_market_proxy,2024-05-08,2024-06-05,cut,cut,True,99.4%,99.4%,0.5%,0.1%,0.000


## Point-outcome accuracy summary

Accuracy answers whether the highest-probability category matched the actual decision. It is included for interpretability, but RPS remains the primary metric because it rewards calibrated uncertainty and uses the ordering of cut, hold, and hike.


In [12]:
outcome_summary = (
    meeting_comparison.groupby("method", as_index=False)
    .agg(
        n_decisions=("correct", "size"),
        point_accuracy=("correct", "mean"),
        mean_prediction_confidence=("prediction_confidence", "mean"),
        mean_rps=("rps", "mean"),
    )
    .sort_values("mean_rps")
)
display(outcome_summary.style.format({
    "point_accuracy": "{:.1%}",
    "mean_prediction_confidence": "{:.1%}",
    "mean_rps": "{:.3f}",
}))


,method,n_decisions,point_accuracy,mean_prediction_confidence,mean_rps
2,quantitative_only_agent,3,100.0%,79.3%,0.072
3,research_grounded_agent,3,100.0%,77.3%,0.083
4,two_year_market_proxy,3,66.7%,99.6%,0.330
1,logistic_regression,3,66.7%,82.9%,0.368
0,historical_frequency,3,33.3%,80.0%,0.616


## Scientific interpretation

Treat this as a smoke-test ablation, not a final result. A defensible study needs a larger post-model-cutoff window, paired uncertainty estimates for RPS differences, document-source ablations, and genuine historical meeting-specific CORRA/OIS probabilities. The key comparison is whether adding cutoff-visible communications improves the research-grounded agent over the otherwise identical quantitative-only agent.
